# 04 – Combine niche model with SeaBee remote sensing

Combines the **niche model** (03) with **SeaBee remote sensing** (shallow water)
into one dataset, which is the input to 05.

- **Niche model**: `nisjemodell_norge_2026_25833.geo.parquet` – rule-based
  classification from light/salinity/substrate/intertidal.
- **SeaBee remote sensing**:
  `marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg`
  – the national regelC habitat product: observed habitat classes
  (ALGAE, ROCK, SEDIMENT, ANGIO, DEEP, ANTHRO) in shallow water.

The rules for how remote sensing overrides the niche model live in
`mnk.rules.classify_seabee` (one rule set, derived from the data model
`nkm_klassifiseringsregler`). ANGIO/ALGAE/ROCK/SEDIMENT are authoritative for
both substrate and light (`nkm_lys`).

**Performance**: the niche model is national (~2.7 M polygons). Only polygons
that actually overlap SeaBee habitat are touched – the rest pass through
unchanged without expensive geometry operations.

In [ ]:
import gc
import io
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.vectorize
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

NICHE_PARQUET = REPO / "nisjemodell_norge_2026_25833.geo.parquet"
SEABEE_GPKG = REPO / "marint_naturkart_habitat_substrat_vegetasjon_regelc_polygoner_uten_rutenett_norge_2024-2025_25833.gpkg"

TORRFALL_PARQUET = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"

# Habitat SeaBee classes that can override the niche model.
# DEEP/ANTHRO do not override (see mnk.rules.SEABEE_RULES).
HABITAT_CLASSES = ["ALGAE", "ROCK", "SEDIMENT", "ANGIO"]

SEABEE_METHOD = "NKM_regelbasert_fjaere"

# Codes in the niche model (nkm_*) -> tokens in mnk.rules.
def _invert(d):
    return {v: k for k, v in d.items()}

LIGHT_MAP = _invert(mnk.rules.NKM_LYS)
SALINITY_MAP = _invert(mnk.rules.NKM_SALINITET)
SUBSTRATE_MAP = _invert(mnk.rules.NKM_SUBSTRAT)
INTERTIDAL_MAP = _invert(mnk.rules.NKM_TOERFALL)

## 1. Load data

In [ ]:
niche = gpd.read_parquet(NICHE_PARQUET)
assert niche.crs.to_epsg() == 25833
print(f"Niche model: {len(niche):,} polygons")

SEABEE_PROB_COLS = ["p_ROCK", "p_SEDIMENT", "p_ALGAE", "p_ANGIO", "p_DEEP", "p_ANTHRO"]
seabee = gpd.read_file(
    SEABEE_GPKG, layer="polygons", columns=["klasse", *SEABEE_PROB_COLS, "p50"]
).to_crs(CRS)
seabee["_usikkerhet"] = mnk.attributes.habitat_usikkerhet(
    seabee[SEABEE_PROB_COLS].to_numpy(), seabee["p50"].to_numpy()
)
seabee_hab = seabee.loc[
    seabee["klasse"].isin(HABITAT_CLASSES), ["klasse", "_usikkerhet", "geometry"]
].reset_index(drop=True)
print(f"SeaBee habitat: {len(seabee_hab):,} polygons")
print(seabee["klasse"].value_counts())
print(seabee_hab["_usikkerhet"].value_counts())

del seabee
gc.collect()

## 2. Find niche model polygons overlapping SeaBee habitat

Spatial index query `sindex.query(..., predicate="intersects")` separates
**affected** polygons from those passed through unchanged. Only affected
polygons go on to the expensive intersection/difference.

In [3]:
# Spatial index query (no global union_all – scales to the whole country).
hit = niche.sindex.query(seabee_hab.geometry, predicate="intersects")
hit_idx = niche.index[np.unique(hit[1])]

affected = niche.loc[hit_idx].copy()
unaffected = niche.drop(index=hit_idx)
print(f"Affected by SeaBee habitat: {len(affected):,}  (unchanged: {len(unaffected):,})")

del niche, hit, hit_idx
gc.collect()

Affected by SeaBee habitat: 144,797  (unchanged: 2,591,688)


0

## 3. Overlay: SeaBee habitat × affected niche polygons

The intersection gives the pieces where remote sensing overrides. Then a
`difference` keeps the part of the affected polygons that lies **outside** the
habitat (unchanged niche model).

In [ ]:
affected = affected.assign(
    _lys=affected[mnk.rules.COL_LYS].map(LIGHT_MAP),
    _sal=affected[mnk.rules.COL_SALINITET].map(SALINITY_MAP),
    _sub=affected[mnk.rules.COL_SUBSTRAT].map(SUBSTRATE_MAP),
    _tf=affected[mnk.rules.COL_TOERFALL].map(INTERTIDAL_MAP),
)
affected["_src"] = np.arange(len(affected))


pairs = gpd.sjoin(
    affected, seabee_hab[["klasse", "_usikkerhet", "geometry"]], how="inner", predicate="intersects"
)
_rgeom = seabee_hab.geometry.to_numpy()[pairs["index_right"].to_numpy()]
_inter = shapely.intersection(pairs.geometry.to_numpy(), _rgeom)
overlaid = pairs.set_geometry(_inter, crs=CRS).drop(columns="index_right")


overlaid = overlaid.explode(index_parts=False)
overlaid = overlaid[overlaid.geom_type == "Polygon"]
overlaid = overlaid[~overlaid.geometry.is_empty & overlaid.geometry.notna()]


_clip = overlaid.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
_src_geom = affected.geometry.to_numpy()
_has = _clip.notna().to_numpy()
_rest = _src_geom.copy()
_rest[_has] = shapely.difference(_src_geom[_has], _clip[_has].to_numpy())
affected_rest = affected.assign(geometry=_rest)
affected_rest = affected_rest[~affected_rest.geometry.is_empty & affected_rest.geometry.notna()]
print(f"Overlay pieces: {len(overlaid):,}  | affected remainder: {len(affected_rest):,}")

del affected, seabee_hab, pairs, _rgeom, _inter, _clip, _src_geom, _has, _rest
gc.collect()

## 4. Apply `classify_seabee` (vectorised over unique combinations)

The number of unique `(klasse, lys, salinitet, substrat, tørrfall)` combinations
is small, so the rules are resolved once per combination and merged back – no
per-polygon Python loop.

In [ ]:
keys = ["klasse", "_lys", "_sal", "_sub", "_tf"]
combos = overlaid[keys].drop_duplicates().reset_index(drop=True)

def resolve(row):
    res = mnk.rules.classify_seabee(
        row["klasse"], row["_lys"], row["_sal"], row["_sub"], row["_tf"]
    )
    return pd.Series(res if res is not None else (pd.NA, pd.NA))

combos[["_nin_code", "_nin_name"]] = combos.apply(resolve, axis=1)
print(combos.to_string())

overlaid = overlaid.merge(combos, on=keys, how="left")
# Drop any non-marine pieces (classify_seabee -> None).
overlaid = overlaid[overlaid["_nin_code"].notna()].copy()

## 5. Update attributes of the remotely sensed pieces

- `nin_kode_fkid` / `NiN_name`: from the remote sensing rule.
- `nkm_prosess`: looked up from `mnk.attributes.NKM_PROSESS`.
- `nkm_lm-dk`: set to the remotely sensed substrate.
- `nkm_lys`: set from `mnk.rules.SEABEE_LYS_OVERRIDE` (bottom observed ⇒ eufotisk).
- `nkm_sa-fa`: fastsittende-alger basistrinn from the SeaBee class — ALGAE ⇒ `ABC`,
  ROCK ⇒ `0B`; other classes leave it NA.
- `metode_klassifisering`: SeaBee method.
- `nkm_usikkerhet`: `lav`/`medium`/`høy` from the SeaBee habitat P-values via
  `mnk.attributes.habitat_usikkerhet` (confidence + class margin); `metode_usikkerhet`
  is `NKM_satelitt_habitat`.
- `datakilde`: SeaBee is authoritative for lys + substrat (`lys_fjernmaalt_seabee`,
  `substrat_fjernmaalt_seabee`); salinitet stays HI and tørrfall stays kartverket.

In [ ]:
# Remote sensing overrides the model substrate. ANGIO (undervannseng) is soft bottom.
SEABEE_SUBSTRATE_NKM = {
    "ALGAE": mnk.rules.NKM_SUBSTRAT["fast"],
    "ROCK": mnk.rules.NKM_SUBSTRAT["fast"],
    "SEDIMENT": mnk.rules.NKM_SUBSTRAT["løst"],
    "ANGIO": mnk.rules.NKM_SUBSTRAT["løst"],
}
SEABEE_LYS_NKM = {k: mnk.rules.NKM_LYS[v] for k, v in mnk.rules.SEABEE_LYS_OVERRIDE.items()}
SEABEE_SA_FA_NKM = {k: mnk.rules.NKM_SA_FA[v] for k, v in mnk.rules.SEABEE_SA_FA_OVERRIDE.items()}

overlaid["nin_kode_fkid"] = overlaid["_nin_code"]
overlaid["NiN_name"] = overlaid["_nin_name"]
overlaid["nkm_prosess"] = overlaid["nin_kode_fkid"].map(mnk.attributes.NKM_PROSESS)
overlaid[mnk.rules.COL_SUBSTRAT] = (
    overlaid["klasse"].map(SEABEE_SUBSTRATE_NKM).fillna(overlaid[mnk.rules.COL_SUBSTRAT])
)
overlaid[mnk.rules.COL_LYS] = (
    overlaid["klasse"].map(SEABEE_LYS_NKM).fillna(overlaid[mnk.rules.COL_LYS])
)
overlaid[mnk.rules.COL_SA_FA] = overlaid["klasse"].map(SEABEE_SA_FA_NKM).astype("string")
overlaid["metode_klassifisering"] = SEABEE_METHOD

overlaid["nkm_usikkerhet"] = overlaid["_usikkerhet"]
overlaid["metode_usikkerhet"] = mnk.attributes.METODE_USIKKERHET_HABITAT

overlaid["datakilde"] = mnk.attributes.build_datakilde(
    np.full(len(overlaid), mnk.attributes.SOURCE_SUBSTRAT_SEABEE),
    overlaid[mnk.rules.COL_TOERFALL],
    lys_token=mnk.attributes.SOURCE_LYS_SEABEE,
)
assert overlaid["nkm_prosess"].notna().all()

## 6. Combine

`combined = unchanged niche model + affected remainder + remotely sensed pieces`.
Only a small minority of polygons went through geometry operations.

`nkm_seabee_klasse` keeps the SeaBee observed class (ALGAE/ROCK/SEDIMENT/ANGIO) for
the remotely sensed pieces, and is NA for the pure nisjemodell pieces.

In [ ]:
unaffected["nkm_seabee_klasse"] = pd.NA
affected_rest["nkm_seabee_klasse"] = pd.NA
overlaid["nkm_seabee_klasse"] = overlaid["klasse"]
unaffected[mnk.rules.COL_SA_FA] = pd.NA
affected_rest[mnk.rules.COL_SA_FA] = pd.NA

OUT_COLS = [c for c in unaffected.columns if not c.startswith("_") and c not in ("nin_int", "nk_fid")]

stamped = gpd.GeoDataFrame(
    pd.concat(
        [unaffected[OUT_COLS], affected_rest[OUT_COLS], overlaid[OUT_COLS]],
        ignore_index=True,
    ),
    crs=CRS,
)
print(f"Stamped (niche ⊕ SeaBee): {len(stamped):,} polygons")
print(stamped["nin_kode_fkid"].value_counts())

del unaffected, affected_rest, overlaid
gc.collect()

## 7. Add tørrfall (NA — awaiting remote sensing)

The niche model + SeaBee do not cover the whole intertidal zone: the 03
nisjemodell clips tørrfall against its own polygons, so intertidal areas with
**missing salinity get no polygon at all**. We add the full tørrfall extent
(`TORRFALL_PARQUET`) back here, for the part not already covered
(`torrfall − stamped`, so the classified intertidal in `stamped` is kept and the
two together preserve the whole tørrfall extent).

These added polygons carry **NA** on every classifying attribute — the four axes
(`nkm_lys`, `nkm_lm-sa`, `nkm_lm-dk`, `nkm_lm-tv`), `nin_kode_fkid` / `NiN_name`,
`nkm_prosess`, `metode_klassifisering`, `nkm_usikkerhet` and `metode_usikkerhet`.
They are *mapped but unclassified*; remote sensing or nisjemodelling fills these
later. Only `opphav_kode = NKM` is set, and the rows are identifiable by a NA
`nin_kode_fkid`.

There is **no deep-sea canvas**: open sea the model does not reach stays an
honest hole rather than being guessed as deep-salt (`NA-MA06`). Possible
brakkvannspoll are likewise not added here — they can be folded in later. The
`torrfall − stamped` difference runs tiled in `mnk.vectorize.coverage_gaps`, so
nothing is unioned/differenced globally.

In [ ]:
TORRFALL_DEFAULTS = {
    "opphav_kode": mnk.attributes.OPPHAV_KODE,
    "nin_kode_fkid": pd.NA,
    "NiN_name": pd.NA,
    "nkm_prosess": pd.NA,
    mnk.rules.COL_LYS: pd.NA,
    mnk.rules.COL_SALINITET: pd.NA,
    mnk.rules.COL_SUBSTRAT: pd.NA,
    mnk.rules.COL_TOERFALL: pd.NA,
    mnk.rules.COL_SA_FA: pd.NA,
    "metode_klassifisering": pd.NA,
    "nkm_usikkerhet": pd.NA,
    "metode_usikkerhet": pd.NA,
    "datakilde": mnk.attributes.SOURCE_TOERFALL_KARTVERKET,
    "nkm_seabee_klasse": pd.NA,
}

torrfall = gpd.read_parquet(TORRFALL_PARQUET).to_crs(CRS)[["geometry"]].reset_index(drop=True)
torrfall["geometry"] = shapely.make_valid(torrfall.geometry.to_numpy())
print(f"Tørrfall: {len(torrfall):,} polygons")
tf_fill = mnk.vectorize.coverage_gaps(torrfall, stamped, tile_size=10_000.0, workers=4)
for col, val in TORRFALL_DEFAULTS.items():
    tf_fill[col] = val
del torrfall
gc.collect()

final = gpd.GeoDataFrame(pd.concat([stamped, tf_fill], ignore_index=True), crs=CRS)
final = final[stamped.columns]
print(f"Coverage: {len(final):,} polygons ({len(tf_fill):,} tørrfall NA-filled)")
print(final["nin_kode_fkid"].value_counts(dropna=False))

del stamped, tf_fill
gc.collect()

## 8. Save

In [ ]:
fname = mnk.utils.to_filename("nisjemodel-remote-sensing", "norge", "2026", CRS.split(":")[1])
parquet_path = REPO / f"{fname}.geo.parquet"

obj_cols = final.select_dtypes("object").columns
final[obj_cols] = final[obj_cols].astype("string")

_meta = io.BytesIO()
final.iloc[:0].to_parquet(_meta)
_meta.seek(0)
geo_meta = pq.read_schema(pa.PythonFile(_meta, mode="r")).metadata[b"geo"]
schema = pa.table(final.iloc[:0].to_arrow(geometry_encoding="WKB")).schema.with_metadata(
    {b"geo": geo_meta}
)

CHUNK = 100_000
with pq.ParquetWriter(parquet_path, schema, compression="snappy") as w:
    for start in range(0, len(final), CHUNK):
        part = pa.table(final.iloc[start : start + CHUNK].to_arrow(geometry_encoding="WKB"))
        w.write_table(part.cast(schema))
        del part
        gc.collect()
print(f"Saved {len(final):,} polygons -> {parquet_path.name}")